In [13]:
from z3 import (
    Bool, BoolRef, ExprRef, Implies, Not, And, Or, Solver,
    Const, Var, Function, DeclareSort, BoolSort, ForAll, Exists,
    is_not, is_and, is_or, is_implies, is_eq, sat, unsat, is_var, is_quantifier
)
from typing import Dict, Tuple, List, Any
from dataclasses import dataclass

In [9]:
# Type aliases for readability
Term = ExprRef
Variable = ExprRef
Formula = ExprRef

# Define a base Sort (type) for all objects in our domain
Object = DeclareSort("Object")

# Constant symbols (denote specific objects)
alice = Const("alice", Object)
bob = Const("bob", Object)
arithmetic = Const("arithmetic", Object)
phoenix = Const("phoenix", Object)
cs221 = Const("cs221", Object)
logic = Const("logic", Object)
two = Const("two", Object)

# Variables (used with quantifiers)
x = Const("x", Object)
y = Const("y", Object)
z = Const("z", Object)

# Functions (map objects to objects)
father = Function("father", Object, Object)
add = Function("add", Object, Object, Object)

# Predicates (map objects to boolean truth values)
Person = Function("Person", Object, BoolSort())
Student = Function("Student", Object, BoolSort())
From = Function("From", Object, Object, BoolSort())
Knows = Function("Knows", Object, Object, BoolSort())
Takes = Function("Takes", Object, Object, BoolSort())
Covers = Function("Covers", Object, Object, BoolSort())
Course = Function("Course", Object, BoolSort())
Concept = Function("Concept", Object, BoolSort())
Even = Function("Even", Object, BoolSort())
GreaterThan = Function("GreaterThan", Object, Object, BoolSort())
Prime = Function("Prime", Object, BoolSort())
Hot = Function("Hot", Object, BoolSort())
City = Function("City", Object, BoolSort())
Place = Function("Place", Object, BoolSort())
Happy = Function("Happy", Object, BoolSort())
#propositional symbol
Snowing = Bool("Snowing") 
Cold = Bool("Cold")



In [11]:
# FOL Solver Question-Answering (ask Function)

def ask(kb: list[Formula], f: Formula) -> str:
    # If KB U {f} = UNSAT, then contradiction
    solver1 = Solver()
    solver1.add(kb + [f])
    if solver1.check() == unsat:
        return "No"

     # If KB U {Not(f)} = UNSAT, then entailment
    solver2 = Solver()
    solver2.add(kb + [Not(f)])
    if  solver2.check() == unsat:
        return "Yes"

    return "I don't know." 

# Motivating Scenario: Reasoning about Alice
kb = []

# Facts
kb.append(Student(alice))
kb.append(From(alice, phoenix))
kb.append(And(Hot(phoenix), City(phoenix)))

# Rules with Universal Quantifiers
kb.append(ForAll([x], Implies(Student(x), Person(x))))
kb.append(ForAll([x], Implies(City(x), Place(x))))
kb.append(Implies(Snowing, Cold))

print("Is it snowing initially? ->", ask(kb, Snowing))  # Expected: I don't know

# Add complex conditional rule:
# If a person is from a hot place and it's snowing, they are unhappy.
kb.append(ForAll([x, y], Implies(
    And(Person(x), From(x, y), Place(y), Hot(y), Snowing),
    Not(Happy(x))
)))

# Fact: Alice is happy
kb.append(Happy(alice))

# Now ask again if it's snowing (Alice being happy forces Snowing to be False!)
print("Is it snowing after knowing Alice is happy? ->", ask(kb, Snowing))  # Expected: No


Is it snowing initially? -> I don't know.
Is it snowing after knowing Alice is happy? -> No


In [33]:
# First-Order Logic Model Definition

@dataclass(frozen=True)
class Interpretation:
    constants: Dict[Const, str]
    functions: Dict[Function, Dict[Any, str]]
    predicates: Dict[Function, Dict[Tuple[str, ...], bool]]

@dataclass(frozen=True)
class FirstOrderLogicModel:
    domain: List[str]
    interpretation: Interpretation

domain = ["o1", "o2", "o3"] # o1: alice, o2: bob, o3: arithmetic

sample_interp = Interpretation(
    constants={
        alice: "o1",
        bob: "o2",
        arithmetic: "o3"
    }, 
    functions={
        father: {
            "o1": "o2" # Father of o1 (alice) is o2 (bob)
        }
    }, 
    predicates={
        Knows: {
            ("o1", "o3"): True, # alice knows arithmetic
            ("o2", "o3"): True, # bob knows arithmetic
        },
        Student: {
            "o1": True, # alice is a student
        }
    })

model = FirstOrderLogicModel(domain, sample_interp)

In [65]:
# Interpreting Terms and Formulas

# Interprets a term (constant, variable, or function application) into a domain object.
# subst: e.g.{ x -> "o1"..} dict mapping from variables to domain objects
def interpret_term(t: Term, w: FirstOrderLogicModel, subst: Dict[Variable, str] = None) -> str:
    if subst is None:
        subst = {}
    if is_var(t) or t in subst:  # Variable (e.g., x)
        return subst[t]
    elif t.num_args() == 0: # Constant symbol (e.g. alice)
        return w.interpretation.constants[t]
    else: # Functions (e.g. father(alice) )
        func, args = t.decl(), t.children()
        func_value = w.interpretation.functions[func]
        args_value = tuple(interpret_term(arg, w, subst) for arg in args)
        if len(args_value) == 1: # Function with one arg (e.g. father(alice))
            return func_value[args_value[0]]
        else:
            return func_value[args_value] # Function with multiple args (e.g. add(a, b))


def interpret_formula(f: Formula, w: FirstOrderLogicModel, subst: Dict[Variable, str] = None) -> bool:
    if subst is None:
        subst = {}
    if is_quantifier(f):
        var = Var(0, f.var_sort(0))
        body = f.body()
        if f.is_forall():
            result = all(interpret_formula(body, w, subst | {var: i}) for i in w.domain)
        elif f.is_exists():
            result = any(interpret_formula(body, w, subst | {var: i}) for i in w.domain)
    elif f.num_args() == 0: # Constant symbol (e.g. alice)
        result = w.interpretation.constants[f]
    elif f.num_args() == 1:
        if is_not(f):  # Negation (e.g., ¬Student(alice))
            result = not interpret_formula(f.arg(0), w, subst)
        else: # Predicate with one argument (e.g. Student(alice))
            predicate, arg = f.decl(), f.arg(0)
            predicate_value = w.interpretation.predicates[predicate]
            arg_value = interpret_term(arg, w, subst)
            result = predicate_value.get(arg_value, False)
    elif f.num_args() == 2:
        arg1 = f.arg(0)
        arg2 = f.arg(1)
        if is_and(f): # e.g. And(Student(alice), Knows(alice, arithmetic))
            result = interpret_formula(arg1, w, subst) and interpret_formula(arg2, w, subst)
        elif is_or(f): 
            result = interpret_formula(arg1, w, subst) or interpret_formula(arg2, w, subst)
        elif is_implies(f):
            result = (not interpret_formula(arg1, w, subst)) or interpret_formula(arg2, w, subst)
        elif is_eq(f):
            result = interpret_formula(arg1, w, subst) == interpret_formula(arg2, w, subst)
        else: 
            # Predicate with multiple args (e.g. Knows(alice, arithmetic))
            predicate, args = f.decl(), f.children()
            predicate_value = w.interpretation.predicates[predicate]
            args_value = tuple(interpret_term(arg, w, subst) for arg in args)
            result = predicate_value.get(args_value, False)
    else:
        raise ValueError(f"Unsupported formula {f} num args {f.num_args()}")

    return result

# Tests
print("Interpret father(alice):", interpret_term(t=father(alice), w=model)) # expect o2
print("Evaluate Student(alice):", interpret_formula(Student(alice), model)) # True
print("Evaluate Student(bob):", interpret_formula(Student(bob), model)) # False
print("Evaluate Knows(alice, arithmetic):", interpret_formula(Knows(alice, arithmetic), model)) # True
print("Evaluate And(Student(alice), Knows(alice, arithmetic)):", interpret_formula(And(Student(alice), Knows(alice, arithmetic)), model)) # True
print("Evaluate ForAll x. Knows(x, arithmetic):", interpret_formula(ForAll([x], Knows(x, arithmetic)), model)) # False (alice & bob know it, but not all domain objects)

Interpret father(alice): o2
Evaluate Student(alice): True
Evaluate Student(bob): False
Evaluate Knows(alice, arithmetic): True
Evaluate And(Student(alice), Knows(alice, arithmetic)): True
Evaluate ForAll x. Knows(x, arithmetic): False


In [66]:
# Variable Substitution Algorithm

def substitute(f: Formula, subst: Dict[Variable, Term]) -> Formula:
    if f in subst: # f is variable (e.g. x)
        return subst[f]
    else:
        new_args = [substitute(f.arg(i), subst) for i in range(f.num_args())]
        return f.decl()(*new_args)

# Substitution Demonstration
formula1 = Knows(x, y)
subst1 = {x: alice, y: cs221}
print("Substitute Knows(x, y) with {x: alice, y: cs221}:")
print(" -> Result:", substitute(formula1, subst1))

formula2 = And(Student(x), Knows(x, y))
subst2 = {x: alice, y: z}
print("\nSubstitute Student(x) & Knows(x, y) with {x: alice, y: z}:")
print(" -> Result:", substitute(formula2, subst2))



Substitute Knows(x, y) with {x: alice, y: cs221}:
 -> Result: Knows(alice, cs221)

Substitute Student(x) & Knows(x, y) with {x: alice, y: z}:
 -> Result: And(Student(alice), Knows(alice, z))


In [67]:
# Unification Algorithm

# Helper to distinguish variables from constants by checking variable names.
def is_variable(f: Formula) -> bool:
    if is_var(f): # Handle Z3 internal De Bruijn variable (如 Var(0))
        return True
    try:
        return f.decl().name() in ["x", "y", "z"]
    except Exception:
        return False

def unify(f1: Formula, f2: Formula, subst: Dict[Variable, Term] = None) -> Dict[Variable, Term]:
    if subst is None:
        subst = {}
        
    if f1 == f2:
        return subst
    elif is_variable(f1):
        subst[f1] = substitute(f2, subst)
    elif is_variable(f2):
        subst[f2] = substitute(f1, subst)
    else:
        if f1.decl() != f2.decl() or f1.num_args() != f2.num_args():
            return None
        for i in range(f1.num_args()):
            if unify(f1.arg(i), f2.arg(i), subst) is None:
                return None
    return subst

# Unification Tests
print("1. Unify Knows(x, y) and Knows(alice, bob):")
print("   ->", unify(Knows(x, y), Knows(alice, bob)))  # Expected: {x: alice, y: bob}

print("2. Unify Knows(alice, y) and Knows(x, z):")
print("   ->", unify(Knows(alice, y), Knows(x, z)))    # Expected: {x: alice, y: z}

print("3. Unify Knows(alice, y) and Knows(bob, z) [Conflict]:")
print("   ->", unify(Knows(alice, y), Knows(bob, z)))  # Expected: None

print(unify(alice, x))

1. Unify Knows(x, y) and Knows(alice, bob):
   -> {x: alice, y: bob}
2. Unify Knows(alice, y) and Knows(x, z):
   -> {x: alice, y: z}
3. Unify Knows(alice, y) and Knows(bob, z) [Conflict]:
   -> None
{x: alice}


In [68]:
# Generalized Modus Ponens & Natural Language Expressions

# Applies Generalized Modus Ponens:
#     Given premises a1', ..., ak' and rule (a1 & ... & ak) -> b,
#     find theta = unify(a1' & ..., a1 & ...) and return substitute(b, theta).
def generalized_modus_ponens(premises: list[Formula], rule: Formula) -> Formula:
    if (not is_quantifier(rule)) or (not is_implies(rule.body())):
        raise ValueError("Rule must be a quantified implication (definite clause).")
    if not premises:
        raise ValueError("premises is empty")
    body = rule.body()
    antecedent = body.arg(0)
    consequent = body.arg(1)
    combined_premises = premises[0]
    for premise in premises[1:]:
        combined_premises = And(combined_premises, premise)
    print(antecedent)
    print(combined_premises)
    subst = unify(combined_premises, antecedent)
    if subst is None:
        return None
    return substitute(consequent, subst)

rule = ForAll([x, y, z], Implies(And(Takes(x, y), Covers(y, z)), Knows(x, z)))
premises = [Takes(alice, cs221), Covers(cs221, logic)]
derived_conclusion = generalized_modus_ponens(premises, rule)
print(derived_conclusion) # Expected: Knows(alice, logic)

And(Takes(Var(2), Var(1)), Covers(Var(1), Var(0)))
And(Takes(alice, cs221), Covers(cs221, logic))
Knows(alice, logic)
